**`ingest_tiles`**

Script examples to import tiling polygons (e.g., for data downloads)

# Configure

In [ ]:
import argparse

from openplaces.io.ingester import Ingester
from openplaces.recipe import get_recipe_by_id
from openplaces.utils import pretty_print

In [ ]:
# Define arguments
parser = argparse.ArgumentParser(description='Ingest tiles using a recipe')
parser.add_argument(
    '--recipe_id',
    help='Identifier of the recipe (e.g., "tiles-osm-2025")',
)
parser.add_argument(
    '--admin_ids',
    help='Administrative unit IDs to ingest (e.g., "US-RI")',
    nargs='*',
)
parser.add_argument(
    '--reprocess',
    help='Reprocess input data from downloaded file',
    action='store_true',
)
parser.add_argument(
    '--redownload',
    help='Redownload input data from original source',
    action='store_true',
)
parser.add_argument(
    '--verbose',
    help='If True, print outputs while processing data',
    action='store_true',
)
parser.add_argument(
    '--keep_unzipped',
    help='If True, keeps unzipped datasets in heap folder after processing',
    action='store_true',
)

# Test arguments

In [ ]:
ARGS_TEST = (
    # Global building recipe #1: OpentilesMap
    # '--recipe_id tile-obm-2025 '
    '--recipe_id US_tile-census-2025_blockgroup '
    # Brunswick, NC (flood/hurricane risk case)
    # '--admin_ids US-NC-BS '
    '--admin_ids US '
    # Processing flags
    '--reprocess '
    # '--redownload '
    '--verbose '
    '--keep_unzipped '
)

# Convert argument string to list of strings
args_list = [x for x in ARGS_TEST.split(' ') if x != '']

# Parse list of arguments
args = parser.parse_args(args_list)

# Display arguments to check if parsing worked as expected
args

In [ ]:
# Show recipe parameters
pretty_print(get_recipe_by_id(args.recipe_id))

# Ingest tile data

In [ ]:
ingester = Ingester(args.recipe_id, args.admin_ids, verbose=args.verbose)

In [ ]:
ingester.ingest(
    reprocess=args.reprocess,
    redownload=args.redownload,
    keep_unzipped=args.keep_unzipped,
)

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
from openplaces.flow import convert_to_script, test_script

COMMIT = True
# If True, writes `.py` scripts to 'scripts/.../'.
# If False, writes a test version of the script to 'scripts/_test/...'

In [ ]:
convert_to_script(commit=COMMIT)

# Test script

In [ ]:
test_script(*args_list, committed=COMMIT)

# Inspect results

## Show full map

In [ ]:
ingester.show_ingested_geometries(fill=False, edgecolor='magenta')

## Show random tile with attributes

In [ ]:
ingester.show_random_entity()

# Link tiles
Spatially linking tiles to `admin_ids` to manage tiled data downloads (to do: move into codebase)

In [ ]:
from openplaces.geo.link import create_entity_link
from openplaces.timing import get_timer

ADMIN_RECIPE_IDS = [
    'admin-gadm-4~1_admin1',
    'admin-gadm-4~1_admin2',
    'admin-gadm-4~1_admin3',
    'US_admin-census-2021_admin2',
    'US_admin-census-2021_admin3',
]

if args.recipe_id == 'tile-obm-2025':
    timer = get_timer(overwrite=True, verbose=True)

    for admin_recipe_id in ADMIN_RECIPE_IDS:
        tile_admin_link = create_entity_link(args.recipe_id, admin_recipe_id)
        timer.mark(f'Tiles linked to: {admin_recipe_id}')